# LeetCode 380 — Insert Delete GetRandom O(1) bằng Python

Link đề: https://leetcode.com/problems/insert-delete-getrandom-o1/ · Độ khó: **Medium** · Chủ đề: Design, Array, Hash Table, Math, Randomized

Skeleton LeetCode:

```python
class RandomizedSet:

    def __init__(self):
        ...

    def insert(self, val: int) -> bool:
        ...

    def remove(self, val: int) -> bool:
        ...

    def getRandom(self) -> int:
        ...


# Your RandomizedSet object will be instantiated and called as such:
# obj = RandomizedSet()
# param_1 = obj.insert(val)
# param_2 = obj.remove(val)
# param_3 = obj.getRandom()
```

Ghi chú: đây là bài **thiết kế lớp** và có yếu tố **ngẫu nhiên**, nên LeetCode không so từng giá trị `getRandom` với đáp án cứng. Trong notebook ta (1) tự dựng bộ chạy chuỗi lệnh mô phỏng cách LeetCode gọi, (2) test `getRandom` bằng **tính chất** (luôn thuộc tập hiện tại) và **thống kê** (mỗi phần tử xuất hiện với tần suất gần bằng nhau, có seed cố định).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thiết kế cấu trúc dữ liệu hỗ trợ `insert`, `remove`, `getRandom`, **mỗi thao tác trung bình `O(1)`**, và `getRandom` phải trả mỗi phần tử với **xác suất bằng nhau**.
2. Nắm các công cụ Python: `list` (`append`, `pop`, chỉ số âm `[-1]`), `dict` (tra cứu/xóa trung bình `O(1)`), module `random` (`choice`, `randrange`, `seed`), gán đồng thời `a, b = b, a`, và cách mô phỏng **chuỗi lệnh** kiểu LeetCode.
3. Đi từ **chỉ dùng list** → **chỉ dùng set** → **list + dict (hoán đổi với phần tử cuối)** → **xóa lười (tombstone) + dựng lại**, và biết mỗi cách vướng ở đâu.
4. Hiểu mấu chốt: `dict` giúp **tìm vị trí** nhanh, `list` giúp **chọn ngẫu nhiên theo chỉ số** nhanh, còn **xóa phần tử bằng cách đổi chỗ với phần tử cuối rồi `pop()`** giữ cả hai luôn đồng bộ.
5. Tránh các lỗi kinh điển: `list.remove`/`pop(i)` tốn `O(n)`, quên cập nhật vị trí của phần tử bị chuyển, sai thứ tự khi xóa đúng phần tử cuối, và `random.choice` trên `set`/`dict`.

## 2. Đề bài & ví dụ

Hãy cài đặt lớp `RandomizedSet` gồm:

- `RandomizedSet()`: khởi tạo đối tượng rỗng.
- `insert(val)`: thêm `val` vào tập nếu **chưa có**; trả `True` nếu thêm được, `False` nếu đã có sẵn.
- `remove(val)`: xóa `val` nếu **đang có**; trả `True` nếu xóa được, `False` nếu không có.
- `getRandom()`: trả về **một phần tử ngẫu nhiên** trong tập hiện tại (đảm bảo lúc gọi tập **không rỗng**); mỗi phần tử phải có **xác suất như nhau**.

Mỗi hàm phải chạy trung bình `O(1)`.

**Ví dụ:**

```
Input
["RandomizedSet", "insert", "remove", "insert", "getRandom", "remove", "insert", "getRandom"]
[[], [1], [2], [2], [], [1], [2], []]
Output
[null, true, false, true, 2, true, false, 2]
```

(`getRandom` lần đầu có thể là `1` hoặc `2`, lần sau chắc chắn là `2`.)

**Ràng buộc:** `-2^31 <= val <= 2^31 - 1`; tối đa `2 * 10^5` lần gọi; khi gọi `getRandom` luôn có ít nhất một phần tử.

In [1]:
import random
import time
from collections import Counter
from typing import List, Optional


def run_ops(cls, ops, args):
    """Mô phỏng cách LeetCode chạy bài thiết kế lớp:
    ops[0] là tên lớp (gọi hàm tạo), các ops còn lại là tên method."""
    obj = None
    out = []
    for i, (op, a) in enumerate(zip(ops, args)):
        if i == 0:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


ops = ["RandomizedSet", "insert", "remove", "insert", "getRandom", "remove", "insert", "getRandom"]
args = [[], [1], [2], [2], [], [1], [2], []]
print(ops)
print(args)

['RandomizedSet', 'insert', 'remove', 'insert', 'getRandom', 'remove', 'insert', 'getRandom']
[[], [1], [2], [2], [], [1], [2], []]


## 3. Kiến thức nền

### 3.1. `list`: thêm cuối nhanh, xóa giữa chậm

`lst.append(x)` và `lst.pop()` (bỏ phần tử **cuối**) đều `O(1)`. Nhưng `x in lst`, `lst.remove(x)` và `lst.pop(i)` (i ở giữa) phải **dò hoặc dồn các phần tử phía sau** nên tốn `O(n)`. `lst[i]` lấy theo chỉ số `O(1)`; `lst[-1]` là phần tử cuối.

In [2]:
lst = [5, 9, 7, 3]
print(lst[-1], lst[0])      # 3 5
lst.append(8)               # O(1)
print(lst)                  # [5, 9, 7, 3, 8]
print(lst.pop())            # 8  (bỏ cuối: O(1))
lst.pop(1)                  # bỏ 9 ở giữa: các phần tử sau phải dồn lên -> O(n)
print(lst)                  # [5, 7, 3]
print(7 in lst)             # True, nhưng phải dò từ đầu: O(n)

3 5
[5, 9, 7, 3, 8]
8
[5, 7, 3]
True


### 3.2. `dict` (hash map): tra cứu, thêm, xóa trung bình `O(1)`

`d[key] = value` thêm/ghi đè, `key in d` kiểm tra tồn tại, `d.pop(key)` xóa và trả `value`, `d[key]` đọc. Cả bốn thao tác **trung bình `O(1)`** nhờ băm khóa. Ta sẽ dùng dict để nhớ "**số `val` đang nằm ở chỉ số nào trong list**".

In [3]:
pos = {}
pos[10] = 0           # số 10 nằm ở chỉ số 0
pos[20] = 1
print(10 in pos, 99 in pos)   # True False
print(pos[20])                # 1
print(pos.pop(10))            # 0  (xóa và trả giá trị cũ)
print(pos)                    # {20: 1}

True False
1
0
{20: 1}


### 3.3. Module `random`

- `random.choice(seq)`: chọn **đều** một phần tử của dãy **có thể đánh chỉ số** (list, tuple, str). **Không** dùng được với `set`/`dict` (không có chỉ số).
- `random.randrange(n)`: số nguyên đều trong `[0, n)`.
- `random.seed(x)`: đặt hạt giống để **kết quả lặp lại được** (rất quan trọng khi test ngẫu nhiên).

In [4]:
random.seed(7)
print([random.randrange(5) for _ in range(8)])
random.seed(7)
print([random.randrange(5) for _ in range(8)])   # giống hệt: cùng seed -> cùng dãy

try:
    random.choice({1, 2, 3})      # set không có chỉ số
except TypeError as e:
    print("Lỗi:", type(e).__name__)

[2, 1, 3, 0, 0, 4, 0, 2]
[2, 1, 3, 0, 0, 4, 0, 2]
Lỗi: TypeError


### 3.4. Hoán đổi và đổi chỗ với phần tử cuối

Python cho phép gán đồng thời: `a[i], a[j] = a[j], a[i]` đổi chỗ hai ô trong một dòng. Ta sẽ dùng thủ thuật: muốn xóa phần tử ở chỉ số `i`, hãy **ghi đè nó bằng phần tử cuối** rồi `pop()` cuối — chỉ tốn `O(1)`.

In [5]:
a = [4, 8, 6, 2]
i = 1                    # muốn xóa a[1] = 8
a[i] = a[-1]             # ghi đè bằng phần tử cuối: [4, 2, 6, 2]
a.pop()                  # bỏ phần tử cuối thừa:     [4, 2, 6]
print(a)                 # thứ tự đổi, nhưng tập hợp đúng là {4, 6, 2}

x, y = 1, 2
x, y = y, x
print(x, y)              # 2 1

[4, 2, 6]
2 1


### 3.5. Mô phỏng "LeetCode gọi lớp"

LeetCode đưa hai dãy: tên lệnh và tham số. Phần tử đầu là hàm tạo, các phần tử sau là method. Hàm `run_ops` ở mục 2 làm đúng việc đó. Ví dụ nhỏ với một lớp đếm:

In [6]:
class Counter2:
    def __init__(self, start): self.v = start
    def add(self, k): self.v += k; return self.v

print(run_ops(Counter2, ["Counter2", "add", "add"], [[10], [5], [1]]))   # [None, 15, 16]

[None, 15, 16]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng một **hộp bốc thăm**: ta bỏ các tờ giấy ghi số vào hộp, lấy ra một tờ cụ thể, hoặc nhắm mắt bốc đại một tờ.

- **Bốc ngẫu nhiên đều** rất dễ nếu các tờ giấy được **xếp thành hàng đánh số 0, 1, 2, ...**: ta chỉ cần chọn một số ngẫu nhiên trong `[0, số_tờ)` rồi lấy tờ ở vị trí đó. Đó chính là `list`.
- Nhưng **thêm/xóa một tờ cụ thể** thì cần biết **nó ở đâu** — nếu hàng không có "mục lục" ta phải dò từng tờ. Mục lục chính là `dict`: "số → vị trí".
- **Xóa** một tờ ở giữa hàng sẽ để lại lỗ hổng. Cách rẻ nhất: **lấy tờ cuối hàng lấp vào lỗ**, rồi bỏ ô cuối; đồng thời **sửa mục lục** cho tờ vừa chuyển.

Tính tay: `items = [10, 20, 30, 40]`, `pos = {10:0, 20:1, 30:2, 40:3}`. Xóa `20`:

1. `i = pos[20] = 1`, `last = items[-1] = 40`.
2. Ghi `items[1] = 40` → `[10, 40, 30, 40]`; sửa `pos[40] = 1`.
3. `items.pop()` → `[10, 40, 30]`; `pos.pop(20)` → `{10:0, 40:1, 30:2}`.

Mọi chỉ số vẫn liên tục `0..2` và mục lục khớp từng phần tử.

## 5. Cách 1 — Brute Force / baseline: chỉ dùng `list`

**Ý tưởng:** lưu mọi thứ trong một list. `insert`: nếu `val` chưa có (`val in items`, dò `O(n)`) thì `append`. `remove`: nếu có thì `items.remove(val)` (`O(n)`). `getRandom`: `random.choice(items)` (`O(1)`).

Đúng hoàn toàn về logic, nhưng `insert` và `remove` là **`O(n)`**, vi phạm yêu cầu `O(1)`.

In [7]:
class RandomizedSetList:
    """Cách 1: chỉ dùng list. insert/remove O(n)."""

    def __init__(self):
        self.items = []

    def insert(self, val: int) -> bool:
        if val in self.items:          # dò toàn bộ list: O(n)
            return False
        self.items.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.items:      # O(n)
            return False
        self.items.remove(val)         # O(n)
        return True

    def getRandom(self) -> int:
        return random.choice(self.items)


random.seed(1)
res = run_ops(RandomizedSetList, ops, args)
print(res)   # getRandom lần đầu là 1 hoặc 2; lần cuối chắc chắn 2
assert res[1:4] == [True, False, True] and res[5:7] == [True, False] and res[7] == 2

[None, True, False, True, 1, True, False, 2]


**Độ phức tạp:** `insert`/`remove` `O(n)`, `getRandom` `O(1)`, bộ nhớ `O(n)`.

**Nút thắt:** hai việc "**có `val` chưa?**" và "**`val` nằm ở đâu để xóa?**" đều phải dò list. Cần một cấu trúc tra cứu theo **giá trị** trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Hai yêu cầu "kéo" ngược nhau:

| Cấu trúc | Tra theo giá trị / xóa | Chọn ngẫu nhiên đều |
|---|---|---|
| `set` / `dict` | `O(1)` | **Không** có chỉ số (phải đổi sang list: `O(n)`) |
| `list` | `O(n)` | `O(1)` bằng `choice` |

**Bước đột phá:** dùng **cả hai** và giữ chúng **đồng bộ**:

- `items`: list các giá trị (để chọn ngẫu nhiên theo chỉ số).
- `pos`: dict `giá trị -> chỉ số trong items` (để tìm nhanh).
- `remove` **không dồn** list mà **lấy phần tử cuối lấp vào chỗ trống**, sau đó `pop()` cuối: chỉ tốn `O(1)`, và vì ta **không quan tâm thứ tự** (tập hợp không có thứ tự), việc đổi thứ tự là vô hại.

Ba bất biến luôn đúng sau mỗi thao tác:

1. `len(items) == len(pos)`.
2. Với mọi `v` trong `pos`: `items[pos[v]] == v`.
3. `items` không có phần tử trùng.

## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (list, mục 5), Cách 2 (set + chuyển sang list mỗi lần `getRandom`), Cách 3 (list + dict, hoán đổi với phần tử cuối — **chính thức**), Cách 4 (xóa lười/tombstone + dựng lại).

### Cách 2 — Chỉ dùng `set`

**Ý tưởng:** `insert` và `remove` dùng `set.add`/`set.remove` (`O(1)`). Với `getRandom`, set không có chỉ số, nên ta **đổi sang list** rồi `choice`: tốn `O(n)`.

- Phần `insert`/`remove` đã nhanh, nhưng `getRandom` lại là `O(n)`: ta chỉ **dời** nút thắt từ chỗ này sang chỗ kia.

In [8]:
class RandomizedSetSet:
    """Cách 2: set. insert/remove O(1), getRandom O(n)."""

    def __init__(self):
        self.s = set()

    def insert(self, val: int) -> bool:
        if val in self.s:
            return False
        self.s.add(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.s:
            return False
        self.s.remove(val)
        return True

    def getRandom(self) -> int:
        return random.choice(list(self.s))   # list(...) copy toàn bộ set: O(n)


random.seed(2)
res = run_ops(RandomizedSetSet, ops, args)
print(res)
assert res[1:4] == [True, False, True] and res[7] == 2

[None, True, False, True, 1, True, False, 2]


**Code nộp LeetCode cho Cách 2** (đúng nhưng `getRandom` `O(n)`, với `2·10^5` lệnh có thể chậm/TLE):

```python
class RandomizedSet:
    def __init__(self):
        self.s = set()

    def insert(self, val: int) -> bool:
        if val in self.s:
            return False
        self.s.add(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.s:
            return False
        self.s.remove(val)
        return True

    def getRandom(self) -> int:
        return random.choice(list(self.s))
```

**Độ phức tạp:** `insert`/`remove` `O(1)` trung bình, `getRandom` `O(n)`, bộ nhớ `O(n)`.

---

### Cách 3 — List + dict, hoán đổi với phần tử cuối (chính thức)

Giải thích từng đoạn:

1. **Khởi tạo:** `self.items = []`, `self.pos = {}`.
2. **`insert(val)`:** nếu `val in self.pos` → `False`. Ngược lại `self.pos[val] = len(self.items)` (vị trí sắp có), rồi `self.items.append(val)`; trả `True`.
3. **`remove(val)`:** nếu `val not in self.pos` → `False`. Ngược lại: `i = self.pos[val]`, `last = self.items[-1]`; `self.items[i] = last`; `self.pos[last] = i`; `self.items.pop()`; `del self.pos[val]`; trả `True`.
4. **`getRandom()`:** `random.choice(self.items)`.

Chú ý dòng `self.pos[last] = i` phải đứng **trước** `del self.pos[val]`: khi `val` chính là phần tử cuối (`last == val`) thì sau hai dòng đầu `pos[val] = i`, rồi `del` xóa nó đi — đúng ý. (Nếu đảo thứ tự sẽ lỗi, xem mục 13.)

In [9]:
class RandomizedSetDemo:
    """Cách 3 (demo): list + dict, remove bằng hoán đổi với phần tử cuối."""

    def __init__(self):
        self.items = []     # các giá trị, để chọn ngẫu nhiên theo chỉ số
        self.pos = {}       # giá trị -> chỉ số trong items

    def insert(self, val: int) -> bool:
        if val in self.pos:
            return False
        self.pos[val] = len(self.items)
        self.items.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val]
        last = self.items[-1]
        self.items[i] = last      # lấp chỗ trống bằng phần tử cuối
        self.pos[last] = i        # sửa "mục lục" cho phần tử vừa chuyển
        self.items.pop()          # bỏ ô cuối thừa
        del self.pos[val]         # xóa mục lục của val
        return True

    def getRandom(self) -> int:
        return random.choice(self.items)


random.seed(3)
res = run_ops(RandomizedSetDemo, ops, args)
print(res)
assert res[1:4] == [True, False, True] and res[5:7] == [True, False] and res[7] == 2

[None, True, False, True, 1, True, False, 2]


**Code nộp LeetCode cho Cách 3:**

```python
import random

class RandomizedSet:
    def __init__(self):
        self.items = []
        self.pos = {}

    def insert(self, val: int) -> bool:
        if val in self.pos:
            return False
        self.pos[val] = len(self.items)
        self.items.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val]
        last = self.items[-1]
        self.items[i] = last
        self.pos[last] = i
        self.items.pop()
        del self.pos[val]
        return True

    def getRandom(self) -> int:
        return random.choice(self.items)
```

**Độ phức tạp:** mỗi thao tác `O(1)` trung bình (dict + thao tác cuối list), bộ nhớ `O(n)`.

---

### Cách 4 — Xóa lười (tombstone) + dựng lại khi quá nhiều "lỗ"

**Ý tưởng khác hẳn:** thay vì lấp lỗ ngay, ta **đánh dấu** ô đã xóa bằng một giá trị đặc biệt `DEAD` (bia mộ), để nguyên trong list. `getRandom` **bốc thử**; nếu trúng `DEAD` thì **bốc lại** (rejection sampling). Để tránh bốc trượt quá nhiều, mỗi khi số `DEAD` vượt **một nửa** độ dài list ta **dọn dẹp** (dựng lại list và dict từ các phần tử còn sống).

- Vì luôn còn **ít nhất một nửa** ô sống nên số lần bốc kỳ vọng `≤ 2`.
- Dọn dẹp tốn `O(n)` nhưng chỉ xảy ra sau **ít nhất `n/2` lần xóa** → chi phí **chia đều (amortized) `O(1)`**.
- Phân phối vẫn **đều** trên các phần tử sống: bốc đều trên mọi ô rồi loại bỏ ô chết ⇒ mỗi ô sống có xác suất bằng nhau.

In [10]:
DEAD = object()   # bia mộ: đối tượng duy nhất, không bao giờ trùng với số nguyên nào


class RandomizedSetLazy:
    """Cách 4: xóa lười + rejection sampling + dọn dẹp khi dead*2 > len."""

    def __init__(self):
        self.items = []
        self.pos = {}
        self.dead = 0

    def insert(self, val: int) -> bool:
        if val in self.pos:
            return False
        self.pos[val] = len(self.items)
        self.items.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        self.items[self.pos.pop(val)] = DEAD      # chỉ đánh dấu
        self.dead += 1
        if self.dead * 2 > len(self.items):       # quá nửa là bia mộ -> dọn
            self.items = [x for x in self.items if x is not DEAD]
            self.pos = {v: i for i, v in enumerate(self.items)}
            self.dead = 0
        return True

    def getRandom(self) -> int:
        while True:
            x = random.choice(self.items)
            if x is not DEAD:
                return x


random.seed(4)
res = run_ops(RandomizedSetLazy, ops, args)
print(res)
assert res[1:4] == [True, False, True] and res[5:7] == [True, False] and res[7] == 2

[None, True, False, True, 1, True, False, 2]


**Code nộp LeetCode cho Cách 4:**

```python
import random

DEAD = object()

class RandomizedSet:
    def __init__(self):
        self.items = []
        self.pos = {}
        self.dead = 0

    def insert(self, val: int) -> bool:
        if val in self.pos:
            return False
        self.pos[val] = len(self.items)
        self.items.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        self.items[self.pos.pop(val)] = DEAD
        self.dead += 1
        if self.dead * 2 > len(self.items):
            self.items = [x for x in self.items if x is not DEAD]
            self.pos = {v: i for i, v in enumerate(self.items)}
            self.dead = 0
        return True

    def getRandom(self) -> int:
        while True:
            x = random.choice(self.items)
            if x is not DEAD:
                return x
```

**Độ phức tạp:** `insert`/`remove` `O(1)` **amortized**, `getRandom` `O(1)` **kỳ vọng** (≤ 2 lần bốc), bộ nhớ `O(n)`. Phức tạp hơn Cách 3 mà không nhanh hơn: chủ yếu để học tư duy "xóa lười" thường gặp trong heap/cache.

## 8. TRACE từng bước

Trace Cách 3 với chuỗi: `insert(10), insert(20), insert(30), insert(40), remove(20), remove(40), insert(50), remove(50)`.

Cột: thao tác → `items` → `pos` → kết quả.

| Thao tác | `items` | `pos` | Kết quả |
|---|---|---|---|
| insert 10 | `[10]` | `{10:0}` | True |
| insert 20 | `[10,20]` | `{10:0,20:1}` | True |
| insert 30 | `[10,20,30]` | `{...,30:2}` | True |
| insert 40 | `[10,20,30,40]` | `{...,40:3}` | True |
| remove 20 | `[10,40,30]` | `{10:0,30:2,40:1}` | True (40 chuyển lên chỉ số 1) |
| remove 40 | `[10,30]` | `{10:0,30:1}` | True (30 chuyển lên) |
| insert 50 | `[10,30,50]` | `{10:0,30:1,50:2}` | True |
| remove 50 | `[10,30]` | `{10:0,30:1}` | True (50 là ô cuối: không cần chuyển ai) |

Hàm dưới in từng bước và **kiểm tra ba bất biến** sau mỗi thao tác.

In [11]:
def check_invariants(items, pos):
    assert len(items) == len(pos)
    assert len(set(items)) == len(items)
    for v, i in pos.items():
        assert items[i] == v


def trace_remove_demo(commands):
    rs = RandomizedSetDemo()
    print(f"{'thao tác':<12}{'kết quả':<9}{'items':<18}pos")
    for op, v in commands:
        r = getattr(rs, op)(v)
        check_invariants(rs.items, rs.pos)
        print(f"{op + ' ' + str(v):<12}{str(r):<9}{str(rs.items):<18}{rs.pos}")


trace_remove_demo([("insert", 10), ("insert", 20), ("insert", 30), ("insert", 40),
                   ("remove", 20), ("remove", 40), ("insert", 50), ("remove", 50),
                   ("remove", 99), ("insert", 30)])

thao tác    kết quả  items             pos
insert 10   True     [10]              {10: 0}
insert 20   True     [10, 20]          {10: 0, 20: 1}
insert 30   True     [10, 20, 30]      {10: 0, 20: 1, 30: 2}
insert 40   True     [10, 20, 30, 40]  {10: 0, 20: 1, 30: 2, 40: 3}
remove 20   True     [10, 40, 30]      {10: 0, 30: 2, 40: 1}
remove 40   True     [10, 30]          {10: 0, 30: 1}
insert 50   True     [10, 30, 50]      {10: 0, 30: 1, 50: 2}
remove 50   True     [10, 30]          {10: 0, 30: 1}
remove 99   False    [10, 30]          {10: 0, 30: 1}
insert 30   False    [10, 30]          {10: 0, 30: 1}


Trace chi tiết **bên trong** một lần `remove` để thấy từng dòng làm gì, kể cả trường hợp xóa chính phần tử cuối:

In [12]:
def trace_remove_inside(items, val):
    pos = {v: i for i, v in enumerate(items)}
    items = list(items)
    print("Trước: items =", items, " pos =", pos, " xóa", val)
    i = pos[val]
    last = items[-1]
    print(f"  i = pos[{val}] = {i};  last = items[-1] = {last}")
    items[i] = last
    print("  items[i] = last      ->", items)
    pos[last] = i
    print("  pos[last] = i        ->", pos)
    items.pop()
    print("  items.pop()          ->", items)
    del pos[val]
    print("  del pos[val]         ->", pos)
    check_invariants(items, pos)
    print()


trace_remove_inside([10, 20, 30, 40], 20)     # xóa ở giữa
trace_remove_inside([10, 20, 30, 40], 40)     # xóa chính phần tử cuối
trace_remove_inside([7], 7)                   # xóa phần tử duy nhất

Trước: items = [10, 20, 30, 40]  pos = {10: 0, 20: 1, 30: 2, 40: 3}  xóa 20
  i = pos[20] = 1;  last = items[-1] = 40
  items[i] = last      -> [10, 40, 30, 40]
  pos[last] = i        -> {10: 0, 20: 1, 30: 2, 40: 1}
  items.pop()          -> [10, 40, 30]
  del pos[val]         -> {10: 0, 30: 2, 40: 1}

Trước: items = [10, 20, 30, 40]  pos = {10: 0, 20: 1, 30: 2, 40: 3}  xóa 40
  i = pos[40] = 3;  last = items[-1] = 40
  items[i] = last      -> [10, 20, 30, 40]
  pos[last] = i        -> {10: 0, 20: 1, 30: 2, 40: 3}
  items.pop()          -> [10, 20, 30]
  del pos[val]         -> {10: 0, 20: 1, 30: 2}

Trước: items = [7]  pos = {7: 0}  xóa 7
  i = pos[7] = 0;  last = items[-1] = 7
  items[i] = last      -> [7]
  pos[last] = i        -> {7: 0}
  items.pop()          -> []
  del pos[val]         -> {}



Quan sát trường hợp xóa `40` (phần tử cuối): `last == val`, nên `items[i] = last` ghi lại chính nó, `pos[last] = i` ghi lại chính nó, `pop()` bỏ nó, `del pos[val]` xóa mục lục — kết quả vẫn đúng **nhờ đúng thứ tự** `pos[last] = i` rồi mới `del pos[val]`.

## 9. Code nộp LeetCode

Phiên bản cuối (Cách 3), đúng tên lớp/method/signature:

In [13]:
class RandomizedSet:

    def __init__(self):
        self.items = []     # các giá trị hiện có
        self.pos = {}       # giá trị -> chỉ số trong items

    def insert(self, val: int) -> bool:
        if val in self.pos:
            return False
        self.pos[val] = len(self.items)
        self.items.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val]
        last = self.items[-1]
        self.items[i] = last
        self.pos[last] = i
        self.items.pop()
        del self.pos[val]
        return True

    def getRandom(self) -> int:
        return random.choice(self.items)


random.seed(11)
print(run_ops(RandomizedSet, ops, args))

[None, True, False, True, 2, True, False, 2]


## 10. Vì sao thuật toán đúng

Dùng **bất biến** (mục 6): sau mỗi thao tác (1) `len(items) == len(pos)`, (2) `items[pos[v]] == v`, (3) không trùng.

- **`insert`:** nếu `val` đã trong `pos` trả `False` và không đổi gì. Nếu chưa: `pos[val] = len(items)` đúng bằng chỉ số `append` sắp tạo ⇒ (2) giữ nguyên; kích thước cả hai cùng tăng 1 ⇒ (1); `val` mới ⇒ (3).
- **`remove`:** nếu không có thì không đổi. Nếu có ở chỉ số `i`: ghi `last` vào `items[i]` và `pos[last] = i` ⇒ `last` giờ nằm đúng nơi mục lục nói. `pop()` bỏ bản sao ở cuối, `del pos[val]` bỏ `val` ⇒ cả hai giảm 1. Nếu `last == val`, ta chỉ xóa phần tử cuối: các bước đều vô hại như trace ở mục 8.
- **`getRandom`:** các chỉ số `0..len-1` **tương ứng 1-1** với phần tử trong tập (bất biến 2, 3), `random.choice` chọn đều trên chỉ số ⇒ mỗi phần tử có xác suất `1/len`.
- **Thời gian:** dict tra/ghi/xóa trung bình `O(1)`, list `append`, `pop()` cuối, `items[i]` đều `O(1)`.

## 11. Phân tích Time / Space Complexity

| Cách | `insert` | `remove` | `getRandom` | Space |
|---|---|---|---|---|
| 1. chỉ list | `O(n)` | `O(n)` | `O(1)` | `O(n)` |
| 2. chỉ set | `O(1)` | `O(1)` | `O(n)` | `O(n)` |
| 3. list + dict (hoán đổi) | `O(1)` | `O(1)` | `O(1)` | `O(n)` |
| 4. tombstone | `O(1)` amortized | `O(1)` amortized | `O(1)` kỳ vọng | `O(n)` (≤ 2 lần số phần tử sống) |

("O(1)" của dict là **trung bình**; trường hợp xấu nhất khi băm đụng độ rất nhiều là `O(n)` — đề cũng chỉ yêu cầu **trung bình**.)

Bộ nhớ: `items` và `pos` mỗi cái `n` phần tử ⇒ `O(n)`.

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề:** các giá trị xác định (`insert`, `remove`) được so cứng; `getRandom` kiểm tra thuộc tập.
2. **Edge case:** xóa phần tử không có, chèn trùng, xóa hết rồi chèn lại, số âm / biên `±2^31`, một phần tử.
3. **Đối chiếu nhiều cách với mô hình chuẩn** (`set` của Python) trên chuỗi lệnh ngẫu nhiên có seed: kết quả `insert`/`remove` phải **khớp từng lệnh**, và `getRandom` luôn thuộc tập.
4. **Thống kê:** mỗi phần tử xuất hiện ≈ `trials/k` lần (sai lệch trong 5 độ lệch chuẩn).

In [14]:
ALL_IMPLS = [RandomizedSetList, RandomizedSetSet, RandomizedSetDemo, RandomizedSetLazy, RandomizedSet]


def run_against_model(cls, num_ops, value_range, seed):
    """Chạy chuỗi lệnh ngẫu nhiên và đối chiếu với set chuẩn."""
    rng = random.Random(seed)
    random.seed(seed + 1)               # seed cho getRandom bên trong lớp
    obj, model = cls(), set()
    for _ in range(num_ops):
        kind = rng.choice(["insert", "remove", "getRandom", "insert"])
        if kind == "insert":
            v = rng.randint(-value_range, value_range)
            assert obj.insert(v) == (v not in model), (cls.__name__, "insert", v)
            model.add(v)
        elif kind == "remove":
            v = rng.randint(-value_range, value_range)
            assert obj.remove(v) == (v in model), (cls.__name__, "remove", v)
            model.discard(v)
        elif model:                      # đề đảm bảo tập không rỗng khi getRandom
            assert obj.getRandom() in model, (cls.__name__, "getRandom")
    return len(model)


# 1) Ví dụ đề (phần xác định)
for cls in ALL_IMPLS:
    random.seed(5)
    r = run_ops(cls, ops, args)
    assert r[1:4] == [True, False, True] and r[5:7] == [True, False]
    assert r[4] in (1, 2) and r[7] == 2
print("Ví dụ đề: OK cho", len(ALL_IMPLS), "cách")

Ví dụ đề: OK cho 5 cách


In [15]:
# 2) Edge cases
for cls in ALL_IMPLS:
    s = cls()
    assert s.remove(5) is False                 # xóa khi rỗng
    assert s.insert(5) is True and s.insert(5) is False
    assert s.getRandom() == 5                   # một phần tử
    assert s.remove(5) is True and s.remove(5) is False
    assert s.insert(5) is True                  # chèn lại sau khi xóa hết
    big, small = 2**31 - 1, -2**31
    assert s.insert(big) and s.insert(small) and s.insert(0) and s.insert(-1)
    assert s.remove(0) and s.remove(big)
    assert {s.getRandom() for _ in range(200)} <= {5, small, -1}
print("Edge cases: OK")

Edge cases: OK


In [16]:
# 3) Đối chiếu với mô hình chuẩn trên chuỗi lệnh ngẫu nhiên (có seed)
for cls in ALL_IMPLS:
    sizes = [run_against_model(cls, 400, vr, seed) for seed in range(30) for vr in (3, 10, 50)]
    print(f"{cls.__name__:<20} OK, kích thước tập cuối trung bình = {sum(sizes) / len(sizes):.1f}")

RandomizedSetList    OK, kích thước tập cuối trung bình = 27.2
RandomizedSetSet     OK, kích thước tập cuối trung bình = 27.2
RandomizedSetDemo    OK, kích thước tập cuối trung bình = 27.2


RandomizedSetLazy    OK, kích thước tập cuối trung bình = 27.2


RandomizedSet        OK, kích thước tập cuối trung bình = 27.2


In [17]:
# 4) Kiểm tra thống kê: mỗi phần tử được chọn đều (sai lệch <= 5 sigma)
def check_uniform(cls, values, trials, seed):
    random.seed(seed)
    s = cls()
    for v in values:
        s.insert(v)
    cnt = Counter(s.getRandom() for _ in range(trials))
    k = len(values)
    p = 1 / k
    expected = trials * p
    sigma = (trials * p * (1 - p)) ** 0.5
    assert set(cnt) == set(values), "có phần tử không bao giờ được chọn / chọn nhầm"
    worst = max(abs(cnt[v] - expected) / sigma for v in values)
    assert worst < 5, (cls.__name__, worst)
    return worst


for cls in ALL_IMPLS:
    w = check_uniform(cls, [10, 20, 30, 40, 50], 50000, seed=2024)
    print(f"{cls.__name__:<20} lệch tối đa {w:.2f} sigma")

RandomizedSetList    lệch tối đa 1.17 sigma
RandomizedSetSet     lệch tối đa 1.17 sigma
RandomizedSetDemo    lệch tối đa 1.17 sigma


RandomizedSetLazy    lệch tối đa 1.17 sigma
RandomizedSet        lệch tối đa 1.17 sigma


In [18]:
# 4b) Đều cả SAU khi đã xóa/chèn nhiều lần (kiểm tra hoán đổi/dọn dẹp không làm lệch)
for cls in ALL_IMPLS:
    random.seed(99)
    s = cls()
    for v in range(20):
        s.insert(v)
    for v in range(0, 20, 2):      # xóa các số chẵn
        s.remove(v)
    s.insert(100); s.insert(101)
    alive = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 100, 101]
    cnt = Counter(s.getRandom() for _ in range(60000))
    assert set(cnt) == set(alive)
    p = 1 / len(alive); exp = 60000 * p; sig = (60000 * p * (1 - p)) ** 0.5
    assert max(abs(c - exp) for c in cnt.values()) < 5 * sig
print("Đều sau khi xóa/chèn: OK")

Đều sau khi xóa/chèn: OK


In [19]:
# 5) Hiệu năng thực tế (chỉ in, không assert): 2*10^5 lệnh với n lớn
def bench(cls, n):
    random.seed(0)
    s = cls()
    t = time.perf_counter()
    for v in range(n):
        s.insert(v)
    for _ in range(n // 2):
        s.getRandom()
    for v in range(0, n, 2):
        s.remove(v)
    return time.perf_counter() - t


for cls, n in [(RandomizedSet, 100000), (RandomizedSetLazy, 100000), (RandomizedSetSet, 4000), (RandomizedSetList, 4000)]:
    print(f"{cls.__name__:<20} n={n:<7} {bench(cls, n):.3f}s")
print("(Cách 1 và 2 chỉ chạy n nhỏ vì quá chậm với n lớn.)")

RandomizedSet        n=100000  0.043s


RandomizedSetLazy    n=100000  0.041s
RandomizedSetSet     n=4000    0.035s


RandomizedSetList    n=4000    0.066s
(Cách 1 và 2 chỉ chạy n nhỏ vì quá chậm với n lớn.)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: xóa bằng `list.remove`/`pop(i)` rồi tưởng vẫn `O(1)`

Sai vì `remove` phải dò và dồn mọi phần tử phía sau ⇒ `O(n)`; thêm nữa **chỉ số của mọi phần tử phía sau đổi**, `pos` sai hàng loạt.

In [20]:
class BadShift:
    def __init__(self):
        self.items, self.pos = [], {}
    def insert(self, val):
        if val in self.pos: return False
        self.pos[val] = len(self.items); self.items.append(val); return True
    def remove(self, val):
        if val not in self.pos: return False
        self.items.pop(self.pos[val])      # dồn list: O(n) và làm lệch pos của các phần tử sau!
        del self.pos[val]
        return True

b = BadShift()
for v in [10, 20, 30]:
    b.insert(v)
b.remove(10)
print(b.items, b.pos)           # items = [20, 30] nhưng pos nói 20->1, 30->2: SAI
try:
    check_invariants(b.items, b.pos)
except AssertionError:
    print("Vi phạm bất biến items[pos[v]] == v")

[20, 30] {20: 1, 30: 2}
Vi phạm bất biến items[pos[v]] == v


**Sửa:** hoán đổi với phần tử cuối (mục 7, Cách 3) — chỉ một phần tử đổi chỉ số, ta cập nhật được ngay.

### Lỗi 2: quên cập nhật `pos[last]`

Chuyển `last` lên chỉ số `i` nhưng không sửa mục lục ⇒ lần sau `remove(last)` lấy chỉ số cũ (đã `pop`) và xóa **nhầm** phần tử khác hoặc báo lỗi.

In [21]:
class BadNoUpdate(BadShift):
    def remove(self, val):
        if val not in self.pos: return False
        i = self.pos[val]
        last = self.items[-1]
        self.items[i] = last           # thiếu: self.pos[last] = i
        self.items.pop()
        del self.pos[val]
        return True

b = BadNoUpdate()
for v in [10, 20, 30]:
    b.insert(v)
b.remove(10)                           # items = [30, 20], nhưng pos[30] vẫn là 2
print(b.items, b.pos)
try:
    b.remove(30)                       # i = 2 -> items[2] không tồn tại
except IndexError as e:
    print("IndexError:", e)

[30, 20] {20: 1, 30: 2}
IndexError: list assignment index out of range


### Lỗi 3: sai thứ tự khi `val` chính là phần tử cuối

Nếu bạn viết `del self.pos[val]` **trước** rồi mới `self.pos[last] = i` thì khi `last == val`, dòng sau **tạo lại** mục lục cho phần tử đã bị xóa ("bóng ma"): `insert(val)` về sau sẽ thấy `val in pos` và trả `False` sai.

In [22]:
class BadOrder(BadShift):
    def remove(self, val):
        if val not in self.pos: return False
        i = self.pos[val]
        last = self.items[-1]
        del self.pos[val]              # xóa trước ...
        self.items[i] = last
        self.pos[last] = i             # ... rồi ghi lại: nếu last == val thì val "sống lại" trong pos
        self.items.pop()
        return True

b = BadOrder()
for v in [10, 20, 30]:
    b.insert(v)
b.remove(30)                           # 30 là phần tử cuối
print(b.items, b.pos)                  # items=[10,20] nhưng pos vẫn có 30
print("insert(30) trả:", b.insert(30), "(đúng phải là True)")

[10, 20] {10: 0, 20: 1, 30: 2}
insert(30) trả: False (đúng phải là True)


**Sửa:** ghi `pos[last] = i` **trước**, `del pos[val]` **sau** (hoặc xử lý riêng trường hợp `i == len(items) - 1`).

### Lỗi 4: dùng `random.choice` trên `set`/`dict`

`set` không có chỉ số ⇒ `TypeError` (mục 3.3). Đổi sang `list(...)` mỗi lần thì đúng nhưng `getRandom` thành `O(n)` (Cách 2). Giữ một `list` song song chính là đáp án.

### Lỗi 5: `insert` quên kiểm tra tồn tại

Nếu `insert` luôn `append`, giá trị trùng xuất hiện **hai lần** trong `items` ⇒ `getRandom` thiên vị (xác suất gấp đôi) và `remove` chỉ xóa được một bản sao.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mọi thao tác `O(1)` + lấy ngẫu nhiên đều"** ⇒ nghĩ ngay **mảng + hash map vị trí**.
- **Xóa ở giữa mảng mà không cần giữ thứ tự** ⇒ **đổi chỗ với phần tử cuối rồi `pop`** (cũng dùng trong nhiều bài khác: xóa trong heap, xóa phần tử khỏi pool...).
- Khi thao tác "nặng" hiếm mà "nhẹ" thường xuyên ⇒ nghĩ **amortized** (Cách 4: dọn dẹp khi quá nửa là rác).
- Nhớ 3 bất biến (`len` bằng nhau, `items[pos[v]] == v`, không trùng) và **assert** chúng trong test của riêng mình.
- Với bài ngẫu nhiên, test bằng **tính chất + thống kê có seed**, không bao giờ so một dãy ngẫu nhiên cụ thể.

## 15. Biến thể của bài

1. **LeetCode 381 — Insert Delete GetRandom O(1) – Duplicates allowed:** cho phép phần tử trùng, `getRandom` chọn theo **tỉ lệ số lần xuất hiện**. Thay `pos` bằng `dict: giá trị -> set các chỉ số`.
2. **`getRandom` có trọng số:** mỗi phần tử có trọng số khác nhau ⇒ dùng mảng tổng tiền tố + tìm nhị phân (xem bài 528 Random Pick with Weight).
3. **Thêm `getMin`/`getMax`** mà vẫn `O(1)` ⇒ cần thêm cấu trúc phụ (ví dụ hai heap + xóa lười, hay cây cân bằng với `O(log n)`).
4. **Lấy `k` phần tử khác nhau ngẫu nhiên** (không lặp): lấy `k` chỉ số khác nhau bằng `random.sample(range(len(items)), k)`.

Ví dụ nhỏ cho biến thể 4:

In [23]:
def get_k_random(rs, k):
    """Trả k phần tử phân biệt, ngẫu nhiên đều, từ RandomizedSet (dùng items)."""
    return [rs.items[i] for i in random.sample(range(len(rs.items)), k)]


random.seed(8)
rs = RandomizedSet()
for v in range(10, 60, 10):
    rs.insert(v)
pick = get_k_random(rs, 3)
print(pick)
assert len(set(pick)) == 3 and set(pick) <= set(rs.items)

[20, 30, 50]


## 16. So sánh các cách làm

| Cách | Ý tưởng | `insert` / `remove` / `getRandom` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. List | mọi thứ trong list | `O(n)` / `O(n)` / `O(1)` | `O(n)` | Chỉ để kiểm chứng logic, dữ liệu rất nhỏ |
| 2. Set | set cho tra/xóa, đổi sang list khi chọn | `O(1)` / `O(1)` / `O(n)` | `O(n)` | `getRandom` hiếm khi gọi |
| 3. List + dict | mục lục vị trí + hoán đổi cuối | `O(1)` / `O(1)` / `O(1)` | `O(n)` | **Đáp án chuẩn** |
| 4. Tombstone | đánh dấu xóa, bốc lại, dọn khi quá nửa | amortized `O(1)` | `O(n)` | Muốn học tư duy xóa lười; khi xóa thật rất đắt |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code. Với `items = [5, 6, 7, 8]`, `pos = {5:0, 6:1, 7:2, 8:3}`, thực hiện `remove(6)` (Cách 3). Viết `items` và `pos` sau đó.

**Bài 2 (tính tay).** Bắt đầu rỗng, thực hiện: `insert(1), insert(2), insert(3), remove(1), remove(3), insert(4), remove(2)`. Ghi bảng `items`/`pos` sau mỗi lệnh và danh sách kết quả trả về.

**Bài 3 (điền TODO).** Hoàn thành lớp `RSTodo` theo Cách 3.

In [24]:
class RSTodo:
    def __init__(self):
        # TODO 1: tạo self.items (list rỗng) và self.pos (dict rỗng)
        pass

    def insert(self, val):
        # TODO 2: nếu val đã có trong pos -> False; ngược lại ghi vị trí, append, True
        return None

    def remove(self, val):
        # TODO 3: lấp chỗ trống bằng phần tử cuối, cập nhật pos[last] TRƯỚC khi xóa pos[val]
        return None

    def getRandom(self):
        # TODO 4: chọn ngẫu nhiên đều trong items
        return None

# KHÔNG tạo đối tượng RSTodo ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết lại `RandomizedSetLazy` (Cách 4) **mà không nhìn lại** mục 7, đặt tên `RSLazy2`; kiểm tra bằng `run_against_model` và `check_uniform`.

**Bài 5 (biến thể / mini challenge).** Viết `RandomizedMultiSet` cho phép trùng (LeetCode 381): `insert(val)` trả `True` nếu `val` **chưa có trước đó**, `remove(val)` xóa **một** bản sao (trả `True` nếu có), `getRandom()` chọn theo **tỉ lệ số lần xuất hiện**. Gợi ý: `pos: dict val -> set(chỉ số)`. Kiểm tra: chèn `1,1,2` rồi `getRandom` phải ra `1` khoảng `2/3` số lần.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `i = pos[6] = 1`, `last = 8`. Sau khi lấp: `items = [5, 8, 7]`; nhớ sửa `pos[8]`. Bài 3: `self.items = []`; `self.pos = {}`.
- **Mức 2 (vừa):** Bài 2: sau `remove(1)` `items = [3, 2]` (3 chuyển lên chỉ số 0). Sau `remove(3)` `items = [2]`. Bài 3: trong `remove`, dùng `i = self.pos[val]; last = self.items[-1]`.
- **Mức 3 (sâu):** Bài 5: khi xóa một bản sao của `val`, lấy **một chỉ số bất kỳ** `i` trong `pos[val]` (`pop` từ set), chuyển phần tử cuối `last` vào `i`: trong `pos[last]` **bỏ chỉ số `len-1`**, **thêm chỉ số `i`** (chú ý trường hợp `last == val`). Khi `pos[val]` rỗng thì xóa khóa.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1
items1 = [5, 6, 7, 8]
pos1 = {5: 0, 6: 1, 7: 2, 8: 3}
i1 = pos1[6]; last1 = items1[-1]
items1[i1] = last1; pos1[last1] = i1; items1.pop(); del pos1[6]
assert items1 == [5, 8, 7] and pos1 == {5: 0, 8: 1, 7: 2}

# Bài 2
rs2 = RandomizedSet()
res2 = [rs2.insert(1), rs2.insert(2), rs2.insert(3), rs2.remove(1), rs2.remove(3), rs2.insert(4), rs2.remove(2)]
assert res2 == [True] * 7
assert rs2.items == [4] and rs2.pos == {4: 0}


# Bài 3
class RSTodoAns:
    def __init__(self):
        self.items = []
        self.pos = {}

    def insert(self, val):
        if val in self.pos:
            return False
        self.pos[val] = len(self.items)
        self.items.append(val)
        return True

    def remove(self, val):
        if val not in self.pos:
            return False
        i = self.pos[val]
        last = self.items[-1]
        self.items[i] = last
        self.pos[last] = i
        self.items.pop()
        del self.pos[val]
        return True

    def getRandom(self):
        return random.choice(self.items)


for seed in range(5):
    run_against_model(RSTodoAns, 300, 8, seed)
assert check_uniform(RSTodoAns, [1, 2, 3, 4], 20000, 3) < 5


# Bài 4
class RSLazy2:
    def __init__(self):
        self.items, self.pos, self.dead = [], {}, 0

    def insert(self, val):
        if val in self.pos:
            return False
        self.pos[val] = len(self.items)
        self.items.append(val)
        return True

    def remove(self, val):
        if val not in self.pos:
            return False
        self.items[self.pos.pop(val)] = DEAD
        self.dead += 1
        if self.dead * 2 > len(self.items):
            self.items = [x for x in self.items if x is not DEAD]
            self.pos = {v: i for i, v in enumerate(self.items)}
            self.dead = 0
        return True

    def getRandom(self):
        while True:
            x = random.choice(self.items)
            if x is not DEAD:
                return x


for seed in range(5):
    run_against_model(RSLazy2, 300, 8, seed)
assert check_uniform(RSLazy2, [1, 2, 3, 4], 20000, 4) < 5


# Bài 5
class RandomizedMultiSet:
    def __init__(self):
        self.items = []
        self.pos = {}                       # val -> set các chỉ số

    def insert(self, val):
        new = val not in self.pos or not self.pos[val]
        self.pos.setdefault(val, set()).add(len(self.items))
        self.items.append(val)
        return new

    def remove(self, val):
        if not self.pos.get(val):
            return False
        i = self.pos[val].pop()             # một chỉ số bất kỳ của val
        last_idx = len(self.items) - 1
        last = self.items[last_idx]
        self.items[i] = last
        if last != val or i != last_idx:
            self.pos[last].discard(last_idx)
            if i != last_idx:
                self.pos[last].add(i)
        self.items.pop()
        return True

    def getRandom(self):
        return random.choice(self.items)


random.seed(123)
ms = RandomizedMultiSet()
assert ms.insert(1) is True and ms.insert(1) is False and ms.insert(2) is True
cnt5 = Counter(ms.getRandom() for _ in range(30000))
assert abs(cnt5[1] / 30000 - 2 / 3) < 0.02
assert ms.remove(1) is True and ms.remove(1) is True and ms.remove(1) is False
assert ms.items == [2] and ms.insert(1) is True
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (list + dict, hoán đổi với phần tử cuối)

```python
import random

class RandomizedSet:
    def __init__(self):
        self.items = []
        self.pos = {}

    def insert(self, val: int) -> bool:
        if val in self.pos:
            return False
        self.pos[val] = len(self.items)
        self.items.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val]
        last = self.items[-1]
        self.items[i] = last
        self.pos[last] = i
        self.items.pop()
        del self.pos[val]
        return True

    def getRandom(self) -> int:
        return random.choice(self.items)
```

**Checklist**

- [ ] `insert` kiểm tra tồn tại **trước** khi thêm (không để trùng)?
- [ ] `pos[val]` được ghi **trước** `append` (bằng `len(items)`)?
- [ ] `remove` dùng hoán đổi với phần tử cuối, **cập nhật `pos[last]` trước** rồi mới `del pos[val]`?
- [ ] Không dùng `list.remove`/`pop(i)`/`x in list` trong đường đi chính?
- [ ] `getRandom` dùng `random.choice` trên **list**, không phải `set`/`dict`?
- [ ] Đã thử: xóa phần tử cuối, xóa phần tử duy nhất, chèn lại sau khi xóa hết, giá trị `±2^31`?
- [ ] Test ngẫu nhiên có **seed** và kiểm tra bằng tính chất/thống kê?